## Process Historical S&P 500 Constituents by Rebalance Cycle
* Divide the historical timeline into rebalance cycles.
* Filter eligible S&P500 constituents as of each rebalance reference date
* Store the constituent set for each cycle for later momentum calculation.
* Method: [S&P500 Momentum Index Methodology](https://www.spglobal.com/spdji/en/documents/methodologies/methodology-sp-momentum-indices.pdf)

In [1]:
import pandas as pd
import numpy as np

In [2]:
sp_membership = pd.read_parquet("../data/raw/sp500_membership.parquet")
sp500 = pd.read_parquet("../data/raw/sp500.parquet")

In [3]:
sp_membership

,permno,start,ending
0,10006,1957-03-01,1984-07-18
1,10030,1957-03-01,1969-01-08
2,10049,1925-12-31,1932-10-01
3,10057,1957-03-01,1992-07-02
4,10078,1992-08-20,2010-01-28
...,...,...,...
2059,93159,2012-07-31,2016-03-29
2060,93246,2021-03-22,2024-12-31
2061,93422,2010-07-01,2015-06-30
2062,93429,2017-03-01,2024-12-31


### Get all rebalance reference date & effective date (1958 - 2024)
According to [S&P500 Momentum Index Methodology](https://www.spglobal.com/spdji/en/documents/methodologies/methodology-sp-momentum-indices.pdf):
> [S&P 500 Momentum] indices rebalance semi-annually, effective after the close on the third Friday of March and September, with a reference date of the last business day of February and August

In [4]:
# effective date
def third_friday(year, month):
    month_days = pd.date_range(
        start=pd.Timestamp(year, month, 1),
        end=pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0),
        freq="D"
    )

    fridays = month_days[month_days.weekday == 4]
    return fridays[2]

# reference date
def last_market_date(trade_dates, year, month):
    trade_dates = pd.DatetimeIndex(trade_dates)

    available = trade_dates[(trade_dates.year == year) &(trade_dates.month == month)]

    if len(available) == 0:
        return pd.NaT

    return available.max()

# actual holding date
def next_market_date(trade_dates, date):
    trade_dates = pd.DatetimeIndex(trade_dates).sort_values()
    later_dates = trade_dates[trade_dates > date]

    if len(later_dates) == 0:
        return pd.NaT

    return later_dates[0]        
        

In [5]:
def get_rebalance_cycles(trade_dates, start_year, end_year):
    records = []
    for year in range(start_year, end_year + 1):
        march_effective = third_friday(year, 3)
        feb_ref = last_market_date(trade_dates, year, 2)
        
        records.append({
            "cycle_id": f"{year}_03",
            "reference_date": feb_ref,
            "effective_date": march_effective,
            "holding_start": next_market_date(trade_dates, march_effective)
        })
        
        sept_effective = third_friday(year, 9)
        aug_ref = last_market_date(trade_dates, year, 8)
        
        records.append({
            "cycle_id": f"{year}_09",
            "reference_date": aug_ref,
            "effective_date": sept_effective,
            "holding_start": next_market_date(trade_dates, sept_effective)
        })
    schedule = pd.DataFrame(records)
    schedule["holding_end"] = schedule["holding_start"].shift(-1).apply(lambda x: trade_dates[trade_dates < x].max())
    return schedule  

In [6]:
trade_days = sp500["date"].unique()

rebalance_cycles = get_rebalance_cycles(trade_days, 1958, 2024)
rebalance_cycles

,cycle_id,reference_date,effective_date,holding_start,holding_end
0,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
1,1958_09,1958-08-29,1958-09-19,1958-09-22,1959-03-20
2,1959_03,1959-02-27,1959-03-20,1959-03-23,1959-09-18
3,1959_09,1959-08-31,1959-09-18,1959-09-21,1960-03-18
4,1960_03,1960-02-29,1960-03-18,1960-03-21,1960-09-16
...,...,...,...,...,...
129,2022_09,2022-08-31,2022-09-16,2022-09-19,2023-03-17
130,2023_03,2023-02-28,2023-03-17,2023-03-20,2023-09-15
131,2023_09,2023-08-31,2023-09-15,2023-09-18,2024-03-15
132,2024_03,2024-02-29,2024-03-15,2024-03-18,2024-09-20


### Get S&P500 Constituents Index Universe for each Rebalance Cycle 
According to [S&P500 Momentum Index Methodology](https://www.spglobal.com/spdji/en/documents/methodologies/methodology-sp-momentum-indices.pdf):

> Index Universe:   
> * At each rebalancing, each index universe is all constituents of the underlying index as of the rebalance reference date.   
> * S&P500 Momentum measurement period: 12 months   

> Eligibility Factors: 
> 1. Have a momentum score
> 2. Stock must have traded at least 150 days during the [12 months] period
> 3. Allow multiple share class     

Summary
1. Get all constituents (index universe) on reference date
2. Filter based on eligibility factors

In [21]:
def filter_trade_days(date, price_history, universe):
    """_summary_
    
        Args:
            date (date): reference date,
            price_history (DataFrame),
            universe (DataFrame),

        Returns:
            DataFrame: universe filtered
    """

    
    # exclude current month, 12 month period
    measure_end = date - pd.offsets.MonthEnd(1)
    measure_start = measure_end - pd.DateOffset(years=1)
    
    # within period, has price
    price_window = price_history.loc[
        (price_history["date"] >= measure_start)
        & (price_history["date"] <= measure_end)
        & (price_history["permno"].isin(universe["permno"]))
        & (price_history["dlyprc"].notna()),
        ["permno", "date"]
    ].drop_duplicates()
    
    # at least 150 days
    trade_days = price_window.groupby("permno").size()
    
    universe = universe.copy()
    universe["trade_days"] = universe["permno"].map(trade_days)
    universe = universe[universe["trade_days"] >= 150].copy()
    universe["measure_start"] = measure_start
    universe["measure_end"] = measure_end
    
    return universe
            
            
def get_eligible_constituents(sp_membership, price_history, rebalance_cycles):

    records = []
    
    for cycle in rebalance_cycles.itertuples():
        date = cycle.reference_date
        if pd.isna(date):
            continue
        
        # get all
        universe = sp_membership[
            (sp_membership["start"] <= date)
            & ((sp_membership["ending"].isna()) | (sp_membership["ending"] >= date))
        ].copy().drop_duplicates(subset="permno")
        
        universe = filter_trade_days(date, price_history, universe)
        
        universe["cycle_id"] = cycle.cycle_id
        universe["reference_date"] = cycle.reference_date
        universe["effective_date"] = cycle.effective_date
        universe["holding_start"] = cycle.holding_start
        universe["holding_end"] = cycle.holding_end

        records.append(universe)
        
    return pd.concat(records, ignore_index=True)
        
        

Download full price_history after 1957-03-04

In [9]:
import wrds
import os
from dotenv import load_dotenv
load_dotenv()
db = wrds.Connection(wrds_username=os.getenv("WRDS_USERNAME"))

Loading library list...
Done


In [ ]:
price_history = db.raw_sql(f"""
    WITH relevant_permnos AS (

        SELECT DISTINCT permno

        FROM crsp.msp500list

        WHERE ending >= DATE '1957-03-04' OR ending IS NULL

    )

    SELECT
        d.permno,
        d.dlycaldt AS date,
        d.dlyprc,
        d.dlyclose,
        d.dlyprcflg,
        d.dlyretx,
        d.dlyret,
        d.dlycumfacpr,
        d.dlyvol,
        d.dlycap

    FROM crsp.dsf_v2 AS d

    INNER JOIN relevant_permnos AS p
        ON d.permno = p.permno

    WHERE d.dlycaldt >= DATE '1956-03-04'
""",
date_cols=["date"]) # get a year earlier in case
price_history


,permno,date,dlyprc,dlyclose,dlyprcflg,dlyretx,dlyret,dlycumfacpr,dlyvol,dlycap
0,10006,1956-03-05,65.75,65.75,TR,0.007663,0.007663,6.0,2400.0,82845.0
1,10006,1956-03-06,64.75,64.75,TR,-0.015209,-0.015209,6.0,1400.0,81585.0
2,10006,1956-03-07,65.0,65.0,TR,0.003861,0.003861,6.0,1000.0,81900.0
3,10006,1956-03-08,65.75,65.75,TR,0.011539,0.011539,6.0,1700.0,82845.0
4,10006,1956-03-09,66.25,66.25,TR,0.007605,0.007605,6.0,3200.0,83475.0
...,...,...,...,...,...,...,...,...,...,...
334011,93436,2025-12-24,485.4,485.4,TR,-0.00033,-0.00033,1.0,40972332.0,1614352542.6
334012,93436,2025-12-26,475.19,475.19,TR,-0.021034,-0.021034,1.0,58470032.0,1580395930.61
334013,93436,2025-12-29,459.64,459.64,TR,-0.032724,-0.032724,1.0,65769882.0,1528679445.16
334014,93436,2025-12-30,454.43,454.43,TR,-0.011335,-0.011335,1.0,58669128.0,1511351928.17


In [17]:
price_history.to_parquet("../data/raw/price_history.parquet")

In [22]:
eligible_constituents = get_eligible_constituents(sp_membership, price_history, rebalance_cycles)
eligible_constituents

,permno,start,ending,trade_days,measure_start,measure_end,cycle_id,reference_date,effective_date,holding_start,holding_end
0,10006,1957-03-01,1984-07-18,253.0,1957-01-31,1958-01-31,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
1,10030,1957-03-01,1969-01-08,253.0,1957-01-31,1958-01-31,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
2,10057,1957-03-01,1992-07-02,253.0,1957-01-31,1958-01-31,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
3,10102,1957-03-01,1975-02-05,253.0,1957-01-31,1958-01-31,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
4,10137,1925-12-31,1976-06-30,253.0,1957-01-31,1958-01-31,1958_03,1958-02-28,1958-03-21,1958-03-24,1958-09-19
...,...,...,...,...,...,...,...,...,...,...,...
66832,93096,2012-12-03,2024-12-31,253.0,2023-07-31,2024-07-31,2024_09,2024-08-30,2024-09-20,2024-09-23,NaT
66833,93132,2018-10-11,2024-12-31,253.0,2023-07-31,2024-07-31,2024_09,2024-08-30,2024-09-20,2024-09-23,NaT
66834,93246,2021-03-22,2024-12-31,253.0,2023-07-31,2024-07-31,2024_09,2024-08-30,2024-09-20,2024-09-23,NaT
66835,93429,2017-03-01,2024-12-31,253.0,2023-07-31,2024-07-31,2024_09,2024-08-30,2024-09-20,2024-09-23,NaT


In [23]:
eligible_constituents.to_parquet("../data/processed/eligible_constituents.parquet")